# Debug gap 14–30 oct. 2025 (raw airqual vs processed)

Objectif : voir si le trou dans Streamlit correspond à **l’absence de raw** (API / ingestion) ou à une perte au **preprocess**.

Prérequis : `.env` à la racine du projet avec `GCP_PROJECT` ; credentials GCP (ADC).

In [1]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

ROOT = Path.cwd().resolve()
if (ROOT / "notebooks").name == "notebooks":
    PROJECT_ROOT = ROOT.parent
else:
    PROJECT_ROOT = ROOT
load_dotenv(PROJECT_ROOT / ".env")

GCP_PROJECT = os.environ["GCP_PROJECT"]
BQ_RAW = os.environ.get("BQ_DATASET_RAW", "breathe_bq_raw")
BQ_PROC = os.environ.get("BQ_DATASET_PROCESSED", "breathe_bq_processed")

START, END = "2025-10-14", "2025-10-30"
client = bigquery.Client(project=GCP_PROJECT)
print(GCP_PROJECT, BQ_RAW, BQ_PROC, f"{START} → {END}")

breathe-air-quality breathe_bq_raw breathe_bq_processed 2025-10-14 → 2025-10-30


## 1. Raw `airqual` — lignes par jour (toutes villes)
Si certains jours sont à 0 partout, le problème est **en amont** (pas de données ingérées pour ces dates).

In [2]:
t_air = f"`{GCP_PROJECT}.{BQ_RAW}.airqual`"
q_air = f"""
SELECT DATE(date) AS day, city, COUNT(*) AS n_rows
FROM {t_air}
WHERE DATE(date) BETWEEN @start AND @end
GROUP BY 1, 2
ORDER BY 1, 2
"""
job = client.query(
    q_air,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("start", "DATE", START),
            bigquery.ScalarQueryParameter("end", "DATE", END),
        ]
    ),
)
df_air = job.result().to_dataframe()
df_air["day"] = pd.to_datetime(df_air["day"])
daily_air = df_air.groupby("day")["n_rows"].sum().rename("n_rows_airqual")
display(daily_air.to_frame())
_have_a = set(pd.to_datetime(daily_air.index).strftime("%Y-%m-%d"))
_all = pd.date_range(START, END, freq="D").strftime("%Y-%m-%d").tolist()
print("Jours sans aucune ligne airqual :", [d for d in _all if d not in _have_a])

/Users/pablorougerie/code/Projets/breathe_project/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_rows_airqual
day,
2025-10-14,35
2025-10-15,35
2025-10-16,33
2025-10-17,33
2025-10-18,33
2025-10-19,33
2025-10-20,33
2025-10-21,34
2025-10-22,34


Jours sans aucune ligne airqual : []


## 2. `processed` — lignes par jour et par ville
Comparer avec le raw : si le raw est OK mais processed vide sur des jours, regarder `drop_na` / features.

In [3]:
t_proc = f"`{GCP_PROJECT}.{BQ_PROC}.processed`"
q_proc = f"""
SELECT DATE(date) AS day, city, COUNT(*) AS n_rows
FROM {t_proc}
WHERE DATE(date) BETWEEN @start AND @end
GROUP BY 1, 2
ORDER BY 1, 2
"""
job = client.query(
    q_proc,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("start", "DATE", START),
            bigquery.ScalarQueryParameter("end", "DATE", END),
        ]
    ),
)
df_proc = job.result().to_dataframe()
df_proc["day"] = pd.to_datetime(df_proc["day"])
daily_proc = df_proc.groupby("day")["n_rows"].sum().rename("n_rows_processed")
display(daily_proc.to_frame())
_have_p = set(pd.to_datetime(daily_proc.index).strftime("%Y-%m-%d"))
_all2 = pd.date_range(START, END, freq="D").strftime("%Y-%m-%d").tolist()
print("Jours sans aucune ligne processed :", [d for d in _all2 if d not in _have_p])

/Users/pablorougerie/code/Projets/breathe_project/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_rows_processed
day,
2025-10-14,6
2025-10-15,6
2025-10-16,6
2025-10-17,6
2025-10-18,6
2025-10-19,6
2025-10-20,6
2025-10-21,6
2025-10-22,6


Jours sans aucune ligne processed : []


## 3. Vue alignée (calendrier vs airqual vs processed)
Une ligne par jour calendaire ; `NaN` = aucune ligne ce jour-là.

In [4]:
cal = pd.DataFrame({"day": pd.date_range(START, END, freq="D")})
cal = cal.merge(daily_air.reset_index(), on="day", how="left")
cal = cal.merge(daily_proc.reset_index(), on="day", how="left")
cal

,day,n_rows_airqual,n_rows_processed
0,2025-10-14,35,6
1,2025-10-15,35,6
2,2025-10-16,33,6
3,2025-10-17,33,6
4,2025-10-18,33,6
5,2025-10-19,33,6
6,2025-10-20,33,6
7,2025-10-21,34,6
8,2025-10-22,34,6
9,2025-10-23,34,6


## 4. (optionnel) Table `monitoring.predictions` sur la même fenêtre
Pour vérifier que ce qui est affiché dans Streamlit correspond bien à ce qui est stocké.

In [5]:
BQ_MON = os.environ.get("BQ_DATASET_MONITORING", "breathe_bq_monitoring")
t_pred = f"`{GCP_PROJECT}.{BQ_MON}.predictions`"
q_pred = f"""
SELECT DATE(date) AS day, city, COUNT(*) AS n
FROM {t_pred}
WHERE DATE(date) BETWEEN @start AND @end
GROUP BY 1, 2
ORDER BY 1, 2
"""
try:
    df_pred = client.query(
        q_pred,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("start", "DATE", START),
                bigquery.ScalarQueryParameter("end", "DATE", END),
            ]
        ),
    ).result().to_dataframe()
    display(df_pred.groupby("day")["n"].sum())
except Exception as e:
    print("predictions :", e)

/Users/pablorougerie/code/Projets/breathe_project/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


day
2025-10-14    6
2025-10-30    6
Name: n, dtype: Int64